# This notebook plays the role of the teacher, ie defines difficulty and create new datasets according to the difficulty

In [4]:
import collections

def create_curriculum_datasets(input_filepath="answers.txt"):
    # We just verify that we only have 5 letters long words
    with open(input_filepath, 'r', encoding='utf-8') as f:
        words = [word.lower() for line in f for word in line.strip().split() if len(word) == 5 and word.isalpha()]
    
    if not words:
        print("Error : No word found.")
        return

    # Want to find the frequence of every letter in this specific dataset
    all_letters = "".join(words)
    letter_counts = collections.Counter(all_letters)
    total_letters = sum(letter_counts.values())
    letter_freqs = {char: count / total_letters for char, count in letter_counts.items()}

    # Let's define word difficulty
    def get_word_score(word):
        unique_letters = set(word) # Keep only unique letters from the word
        
        freq_score = sum(letter_freqs[char] for char in unique_letters)
        
        # We want to penalise duplicates - see the report for further details
        duplicates = len(word) - len(unique_letters)
        penalty = duplicates * 0.15 # random choice
        
        return freq_score - penalty

    scored_words = [(word, get_word_score(word)) for word in words]
    
    # Here we want to implement classic Curicculum learning, then we sort from easy to hard
    scored_words.sort(key=lambda x: x[1], reverse=True)
    sorted_words = [word for word, score in scored_words]

    # Let's divide the dataset into 3 parts to create 3 levels of difficulty
    n = len(sorted_words)
    easy = sorted_words[:n//3]
    medium = sorted_words[n//3:2*n//3]
    hard = sorted_words[2*n//3:]

    datasets = {
        "dataset_1_easy.txt": easy,
        "dataset_2_medium.txt": medium,
        "dataset_3_hard.txt": hard
    }

    for filename, dataset in datasets.items():
        with open(filename, 'w', encoding='utf-8') as f:
            for w in dataset:
                f.write(w + '\n')
                
    print(f"Done ! Datasets created :")
    print(f"- {len(easy)} easy words (ex: {', '.join(easy[:5])})")
    print(f"- {len(medium)} medium words (ex: {', '.join(medium[:5])})")
    print(f"- {len(hard)} hard words (ex: {', '.join(hard[-5:])})")

In [5]:
create_curriculum_datasets()

Done ! Datasets created :
- 771 easy words (ex: alert, alter, later, arose, irate)
- 772 medium words (ex: evict, rhyme, stork, sloth, fetid)
- 772 hard words (ex: civic, puppy, vivid, mummy, mamma)


## Test to have smaller datasets

In [6]:
import collections

def create_curriculum_subsets(input_filepath="wordle_actual.txt", subset_size=479):
    with open(input_filepath, 'r', encoding='utf-8') as f:
        words = [word.lower() for line in f for word in line.strip().split() if len(word) == 5 and word.isalpha()]
    
    if not words:
        print("Error, no word found.")
        return

    all_letters = "".join(words)
    letter_counts = collections.Counter(all_letters)
    total_letters = sum(letter_counts.values())
    letter_freqs = {char: count / total_letters for char, count in letter_counts.items()}

    def get_word_score(word):
        unique_letters = set(word)
        freq_score = sum(letter_freqs[char] for char in unique_letters)
        duplicates = len(word) - len(unique_letters)
        penalty = duplicates * 0.15 
        return freq_score - penalty

    scored_words = [(word, get_word_score(word)) for word in words]
    scored_words.sort(key=lambda x: x[1], reverse=True)
    sorted_words = [word for word, score in scored_words]

    n = len(sorted_words)
    full_easy = sorted_words[:n//3]
    full_medium = sorted_words[n//3:2*n//3]
    full_hard = sorted_words[2*n//3:]

    def extract_subset(source_list, size):
        if len(source_list) < size:
            return source_list
        
        step = len(source_list) / size
        return [source_list[int(i * step)] for i in range(size)]

    subsets = {
        "subset_1_easy_479.txt": extract_subset(full_easy, subset_size),
        "subset_2_medium_479.txt": extract_subset(full_medium, subset_size),
        "subset_3_hard_479.txt": extract_subset(full_hard, subset_size)
    }

    for filename, data in subsets.items():
        with open(filename, 'w', encoding='utf-8') as f:
            for w in data:
                f.write(w + '\n')
                
    print(f"Done ! 3 subsets of {subset_size} words created.")
    for name, data in subsets.items():
        print(f"- {name} : de '{data[0]}' à '{data[-1]}'")

create_curriculum_subsets()

Done ! 3 subsets of 479 words created.
- subset_1_easy_479.txt : de 'arose' à 'dirge'
- subset_2_medium_479.txt : de 'thous' à 'scree'
- subset_3_hard_479.txt : de 'prees' à 'fuffy'
